In [170]:
import torch
import torch.nn as nn
import re
import torch.optim as optim
from rich.progress import track
import torch.nn.functional as F

In [171]:
text = """
  From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine where abundance lies,
  Thy self thy foe, to thy sweet self too cruel:
  Thou that art now the world's fresh ornament,
  And only herald to the gaudy spring,
  Within thine own bud buriest thy content,
  And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee."""

data = text.lower().split('\n')

text = re.sub(r'[^a-zA-Z\s]','',text)

words = text.lower().split()

wti = {c:i for i,c in enumerate(words)}
itw = {i:c for i,c in enumerate(words)}

seq = 6

X = []
y = []

for i in range(len(words)-seq):
  X.append([wti[w] for w in words[i:i+seq]])
  y.append([wti[words[i+seq]]])

X = torch.tensor(X)
y = torch.tensor(y)

In [172]:
class MyLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        self.fh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.fx = nn.Linear(input_size,  hidden_size, bias=False)
        self.fb = nn.Parameter(torch.randn(hidden_size))

        self.ih = nn.Linear(hidden_size, hidden_size, bias=False)
        self.ix = nn.Linear(input_size,  hidden_size, bias=False)
        self.ib = nn.Parameter(torch.randn(hidden_size))

        self.gh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.gx = nn.Linear(input_size,  hidden_size, bias=False)
        self.gb = nn.Parameter(torch.randn(hidden_size))

        self.oh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.ox = nn.Linear(input_size,  hidden_size, bias=False)
        self.ob = nn.Parameter(torch.randn(hidden_size))

    def compute_forgot(self,h,x):
        # print(h.shape,x.shape)
        return torch.sigmoid(self.fh(h) + self.fx(x) + self.fb)

    def compute_input(self,h,x):
        return torch.sigmoid(self.ih(h) + self.ix(x) + self.ib)

    def compute_gate(self,h,x):
        return torch.tanh(self.gh(h) + self.gx(x) + self.gb)

    def compute_output(self,h,x):
        return torch.sigmoid(self.oh(h) + self.ox(x) + self.ob)

    def lstm_cell(self,x,h_prev, c_prev):
        f = self.compute_forgot(h_prev,x)
        i = self.compute_input(h_prev, x)
        g = self.compute_gate(h_prev,x)
        o = self.compute_output(h_prev, x)

        cell_state   = (c_prev * f)+(i * g)
        hidden_state = o *torch.tanh(cell_state)

        return hidden_state, cell_state

    def forward(self, x, h0=None, c0=None):
        # x shape: (seq_len, batch, input_size)
        seq_len, batch, _ = x.shape
        # print(f"x.shape: {x.shape}")
        h = h0 if h0 is not None else torch.zeros(batch, self.hidden_size)
        c = c0 if c0 is not None else torch.zeros(batch, self.hidden_size)
        # print(x.shape,h.shape)

        outputs = []
        for t in range(seq_len):
            h, c = self.lstm_cell(x[t], h, c)
            outputs.append(h.unsqueeze(0))
        # print(torch.cat(outputs, dim=0))
        return torch.cat(outputs, dim=0)

In [173]:
embedding = nn.Embedding(len(words),100)

model = MyLSTM(100,100)

ann_layer = nn.Linear(100,len(words))

In [174]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(model.parameters())+list(ann_layer.parameters()), lr=0.01)

In [175]:
data_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X,y), batch_size=10, drop_last=True,shuffle=True)

In [176]:
for epoch in track(range(100),description="Training Data: "):
  for X_train,y_true in data_loader:
    embeddings = embedding(X_train)
    # Removed .squeeze(-1) to maintain (batch, seq, feature) shape
    output = model.forward(embeddings.permute(1,0,2))
    y_pred = ann_layer(output[-1])
    loss = loss_fn(y_pred,y_true.squeeze())
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()

Output()

In [210]:
def generate_response(model,text,max_tokens=20, temperature=0.8, top_k=0, top_p=0.75):

  current_input_words = text.split()
  final_data_words = list(current_input_words)

  print(text)

  model.eval()
  with torch.no_grad():
    for i in range(max_tokens):
      model_input_words = current_input_words[-seq:]
      data = [wti[chunk.lower().strip()] for chunk in model_input_words]
      data = torch.tensor([data])

      embeddings = embedding(data)
      output = model.forward(embeddings)

      output = ann_layer(output[:,-1,:])

      probabilities = F.softmax(output / temperature, dim=-1)
      # print(probabilities)
      if top_p < 1.0:
          sorted_probs, sorted_indices = torch.sort(probabilities, descending=True)
          cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
          num_to_keep = (cumulative_probs < top_p).sum(dim=-1) + 1
          # print(num_to_keep)
          mask = torch.arange(probabilities.shape[-1], device=probabilities.device).unsqueeze(0) < num_to_keep.unsqueeze(1)
          # print(probabilities.shape[-1])
          filtered_sorted_probs = sorted_probs * mask
          # print(filtered_sorted_probs)

          probabilities = torch.zeros_like(probabilities).scatter_(-1, sorted_indices, filtered_sorted_probs)
          probabilities = probabilities / probabilities.sum(dim=-1, keepdim=True)
      elif top_k > 0:
          values, indices = torch.topk(probabilities, k=top_k, dim=-1)
          probabilities = torch.zeros_like(probabilities).scatter_(-1, indices, values)
          probabilities = probabilities / probabilities.sum(dim=-1, keepdim=True)
      # print(probabilities)



      word_idx = torch.multinomial(probabilities, 1).item()
      predicted_word = itw[word_idx]

      final_data_words.append(predicted_word)
      current_input_words.append(predicted_word)

  return ' '.join(final_data_words)

# text = input('enter text: ').strip().lower()

generate_response(model,"from",max_tokens=30, temperature=1, top_k=0, top_p=0.75)

from


'from that art now the worlds due by time to thine own bright eyes feedst thy foe to the world or making due by time decease his memory but as the'